# 01 — Data Cleaning & Preprocessing
**Goal:** Load the raw Telco dataset, fix data types, handle missing values, and save a clean version.

**Dataset:** Telco Customer Churn — 7,043 customers, 21 features

**Key findings to look for:**
- `TotalCharges` has spaces stored as strings (→ NaN after `pd.to_numeric`)
- New customers (tenure=0) have no total charges → fill with 0
- Churn is `Yes/No` → encode as 1/0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load raw data
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Data types & memory ────────────────────────────────
df.info()

In [ ]:
# ── Missing values ─────────────────────────────────────
print('\nMissing values before fix:')
print(df.isnull().sum())

# Fix TotalCharges: spaces → NaN → 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'\nTotalCharges NaN count: {df["TotalCharges"].isna().sum()}')
df['TotalCharges'].fillna(0, inplace=True)

print('\nMissing values after fix:')
print(df.isnull().sum())

In [ ]:
# ── Duplicates ─────────────────────────────────────────
print(f'Duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)

# ── Encode target ──────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f'\nChurn distribution:\n{df["Churn"].value_counts()}')
print(f'Churn rate: {df["Churn"].mean():.2%}')

In [ ]:
# ── Strip whitespace from string cols ──────────────────
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# ── Quick stats on numeric columns ─────────────────────
df.describe()

In [ ]:
# ── Visualise: TotalCharges distribution ──────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['tenure'], bins=30, color='#6366f1', edgecolor='white', alpha=0.8)
axes[0].set_title('Tenure Distribution'); axes[0].set_xlabel('Months')

axes[1].hist(df['MonthlyCharges'], bins=30, color='#f43f5e', edgecolor='white', alpha=0.8)
axes[1].set_title('Monthly Charges Distribution')

axes[2].hist(df['TotalCharges'], bins=30, color='#22c55e', edgecolor='white', alpha=0.8)
axes[2].set_title('Total Charges Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# ── Data Dictionary ────────────────────────────────────
data_dict = pd.DataFrame({
    'Feature': df.columns,
    'Dtype': df.dtypes.values,
    'Unique Values': df.nunique().values,
    'Sample': [str(df[c].iloc[0]) for c in df.columns]
})
data_dict

In [ ]:
# ── Save cleaned data ──────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print(f'✅ Saved cleaned data: {df.shape}')